# Refined Feature-Rich MF6 Workflow

This notebook builds a more realistic synthetic MODFLOW 6 example with a refined Voronoi grid.
Instead of a toy four-cell mesh, it uses `TriangleGrid` to define a rectangular domain, a broad interior refinement zone, a stream-corridor refinement zone, and a finer lake refinement zone before converting the mesh to `VoronoiGridPlus`.

The workflow then adds:

- CHD on the eastern boundary
- DRN from a polygon in the northwest
- GHB from a polygon in the southwest
- recharge zones and UZF
- LAK from a lake polygon
- SFR from a stream line
- region and group registration for later querying


In [ ]:
from pathlib import Path
import shutil

import geopandas as gpd
import numpy as np
from shapely.geometry import LineString, Polygon

import myflopy as mf
from myflopy.modflow.mf6.drn import DRNFromVector
from myflopy.modflow.mf6.ghb import GHBFromVector
from myflopy.modflow.mf6.lakes import LAKBuilder
from myflopy.modflow.mf6.recharge import RCHFromVector
from myflopy import SFRBuilder
from myflopy import ModelContext, UZFBuilder
from myflopy.modflow.mf6.simulation.discretization import DisvGrid, TemporalDiscretization
from myflopy.modflow.mf6.simulation.packages import (
    CHD,
    InitialConditions,
    KFlow,
    OutputControl,
    Recharge,
    Storage,
)


In [ ]:
if Path.cwd().name == "notebooks" and Path.cwd().parent.name == "mf6":
    examples_root = Path.cwd().parent
else:
    examples_root = Path.cwd() / "examples" / "mf6"

workspace = examples_root / "artifacts" / "refined_feature_rich_model_workflow"
if workspace.exists():
    shutil.rmtree(workspace, ignore_errors=True)
workspace.mkdir(parents=True, exist_ok=True)

workspace

In [ ]:
def write_gpkg(path: Path, gdf: gpd.GeoDataFrame) -> Path:
    gdf.to_file(path, driver="GPKG")
    return path


def build_refined_vor(model_ws: Path) -> mf.VoronoiGridPlus:
    triangle_ws = model_ws / "triangle_build"
    triangle_ws.mkdir(parents=True, exist_ok=True)
    tri = mf.TriangleGrid(model_ws=str(triangle_ws))
    tri.set_domain_rectangle(x_dist=2000, y_dist=1500, origin=(0, 0))
    tri.add_region_rectangle(
        origin=(150, 150),
        x_dist=1700,
        y_dist=1200,
        max_area=60000,
        label="mid_refine",
    )
    tri.add_region_polygon(
        LineString([(120, 1100), (1880, 500)]).buffer(75),
        max_area=12000,
        label="stream_refine",
        priority=2,
        source="line",
    )
    tri.add_region_circle(
        center_coords=(700, 500),
        radius=180,
        max_area=5000,
        label="lake_refine",
        priority=3,
    )
    tri.build(verbose=False)

    vor = mf.VoronoiGridPlus(tri)
    top = 135.0 - (0.012 * np.asarray(vor.centroids_x)) + (0.006 * np.asarray(vor.centroids_y))
    bottom = top - 45.0
    vor.gdf_topbtm = gpd.GeoDataFrame(
        {
            "geometry": vor.gdf_vorPolys.geometry,
            0: top,
            1: bottom,
        },
        geometry="geometry",
        crs=vor.crs,
    )
    return vor, tri


vor, tri = build_refined_vor(workspace)
len(vor.gdf_vorPolys)


In [ ]:
tri.preview_regions()[["label", "max_area", "claim_area", "priority"]]

In [ ]:
vor.plot2d()

In [ ]:
top = vor.gdf_topbtm[0].to_list()
bottom = [vor.gdf_topbtm[1].to_list()]
strt = (vor.gdf_topbtm[0] - 8.0).to_list()
k = (40.0 - (0.015 * np.asarray(vor.centroids_x))).clip(min=5.0).tolist()

model = mf.SimulationBase(name="refined_demo", mf_folder_path=workspace, vor=vor, nper=2)
DisvGrid(vor=vor, model=model, top=top, bottom=bottom, nlay=1)
TemporalDiscretization(model=model, per_len=1, num_steps=1, multiplier=1.0)
InitialConditions(model=model, vor=vor, nlay=1, strt=strt)
KFlow(model=model, k=k, k33_vert=[value * 0.1 for value in k], save_specific_discharge=False)
Storage(model=model, sto_steady={0: True}, sto_transient={1: True})
OutputControl(model=model)

east_strip = Polygon([(1850, 0), (2000, 0), (2000, 1500), (1850, 1500)])
east_cells = sorted(set(vor.get_vor_cells_as_series(east_strip).iloc[0]))
chd_spd = {
    0: [[(0, cell), 104.0] for cell in east_cells],
    1: [[(0, cell), 103.5] for cell in east_cells],
}
CHD(model=model, stress_period_data=chd_spd)

len(east_cells)

In [ ]:
drain_path = write_gpkg(
    workspace / "drain.gpkg",
    gpd.GeoDataFrame(
        {
            "name": ["northwest_drain"],
            "height": [2.0],
            "cond": [1500.0],
            "layer": [1],
            "min_elev": [85.0],
        },
        geometry=[Polygon([(0, 1100), (700, 1100), (700, 1500), (0, 1500)])],
        crs=vor.crs,
    ),
)

ghb_path = write_gpkg(
    workspace / "ghb.gpkg",
    gpd.GeoDataFrame(
        {
            "name": ["southwest_ghb"],
            "elev": [96.0],
            "height": [0.0],
            "cond": [2200.0],
            "layer": [1],
            "min_elev": [80.0],
        },
        geometry=[Polygon([(0, 0), (550, 0), (550, 450), (0, 450)])],
        crs=vor.crs,
    ),
)

recharge_path = write_gpkg(
    workspace / "recharge.gpkg",
    gpd.GeoDataFrame(
        {
            "zone": ["uplands", "lowlands"],
            "rch_0": [0.004, 0.0025],
            "rch_1": [0.005, 0.003],
        },
        geometry=[
            Polygon([(0, 700), (2000, 700), (2000, 1500), (0, 1500)]),
            Polygon([(0, 0), (2000, 0), (2000, 700), (0, 700)]),
        ],
        crs=vor.crs,
    ),
)

lake_path = write_gpkg(
    workspace / "lake.gpkg",
    gpd.GeoDataFrame(
        {"name": ["lake_0"]},
        geometry=[Polygon([(520, 340), (860, 340), (860, 670), (520, 670)])],
        crs=vor.crs,
    ),
)

stream_path = write_gpkg(
    workspace / "stream.gpkg",
    gpd.GeoDataFrame(
        {"name": ["stream_0"]},
        geometry=[LineString([(160, 1120), (650, 930), (1120, 760), (1760, 520)])],
        crs=vor.crs,
    ),
)


In [ ]:
drn_builder = DRNFromVector(model=model, vor=vor, shp_gpkg=drain_path, uid="name", idomain=[1] * vor.ncpl)
drn_dict = drn_builder.from_vector(
    edges_only=True,
    register_regions=True,
    region_name_prefix="drn_group",
    combined_region_name="all_drains",
    region_tags=["drn"],
    overwrite_regions=True,
)
mf.modflow.mf6.simulation.packages.Drains(model=model, stress_period_data=drn_dict)

ghb_builder = GHBFromVector(model=model, vor=vor, shp_gpkg=ghb_path, uid="name", idomain=[1] * vor.ncpl)
ghb_dict = ghb_builder.from_vector(
    register_regions=True,
    region_name_prefix="ghb_group",
    combined_region_name="all_ghb",
    region_tags=["ghb"],
    overwrite_regions=True,
)
mf.modflow.mf6.simulation.packages.GHB(model=model, stress_period_data=ghb_dict)

recharge_builder = RCHFromVector(
    model=model,
    vor=vor,
    shp_gpkg=recharge_path,
    uid="zone",
    rch_fields=["rch_0", "rch_1"],
    rch_fields_to_pers=[0, 1],
    background_rch=0.0,
    grid_type="disv",
    limit_to_k33=False,
)
rch_dict = recharge_builder.from_vector(
    register_regions=True,
    region_name_prefix="rch_zone",
    combined_region_name="all_rch",
    region_tags=["rch"],
    overwrite_regions=True,
)
Recharge(model=model, vor=vor, rch_dict=rch_dict)

uzf_context = ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain))
uzf_cells = [(0, cell) for cell in range(vor.ncpl)]
uzf_finf = {period: [{tuple(row[0]): row[1] for row in rows}.get(cellid, 0.0) for cellid in uzf_cells] for period, rows in rch_dict.items()}
uzf = UZFBuilder(
    context=uzf_context,
    nper=model.nper,
    cells=uzf_cells,
    vks=0.75,
    thtr=0.1,
    thts=0.3,
    thti=0.2,
    finf=uzf_finf,
)
uzf.build().build(model.gwf)
model.add_region_from_cells("uzf_all", uzf.uzf_cells, category="boundary", package="uzf", tags=["uzf"], overwrite=True)

lak = LAKBuilder(
    context=ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain)),
    nper=model.nper,
    lakes=[lake_path],
    lake_id_field="name",
    starting_stage=110.0,
    lake_bottom=90.0,
    bed_leakance=0.05,
    connection_modes="automatic",
    status="ACTIVE",
    mover=False,
)
lak.build().build(model.gwf)
for lake_id, cells in lak.lake_cells.items():
    model.add_region_from_cells(f"lake_zone_{lake_id}", [(0, cell) for cell in cells], category="boundary", package="lak", tags=["lak"], geometry=lak.lake_table.loc[lake_id].geometry, overwrite=True)
model.add_region_from_cells("all_lakes", [(0, cell) for cells in lak.lake_cells.values() for cell in cells], category="boundary", package="lak", tags=["lak"], overwrite=True)

sfr = SFRBuilder(
    context=ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain)),
    nper=model.nper,
    streams=[stream_path],
    width=25.0,
    gradient=0.001,
    roughness=0.03,
    streambed_k=3.0,
    streambed_thickness=2.0,
)
sfr.build().build(model.gwf)
model.add_region_from_cells("all_streams", [(0, cell) for cells in sfr.stream_cells.values() for cell in cells], category="boundary", package="sfr", tags=["sfr"], overwrite=True)

model.add_group(
    "boundary_features",
    members=["all_drains", "all_ghb", "all_rch", "uzf_all", "all_lakes", "all_streams"],
)


In [ ]:
success, _ = model.run_simulation()
success

In [ ]:
model.list_regions().head()

In [ ]:
resolved_cells, trace = model.resolve_region_cells_with_trace("boundary_features")
len(resolved_cells), list(trace.items())[:5]

In [ ]:
model.region_heads("boundary_features", per=1)

In [ ]:
model.gwf.chd.plot()

In [ ]:
model.plot.map(per=1)